# 03 - The Factor Zoo

Phase 2 builds a library of cross-sectional factors behind one interface. Each
factor is a `Factor` (see `qer.factors.base`): it sets a `name` and a
`direction` (+1, or -1 if the raw score must be sign-flipped to mean "high =
expected high return") and implements a **vectorised** `compute_panel(loader)`
that returns the whole `date x ticker` matrix in one pass.

This notebook is the "here is each factor, concretely" tour: build each one,
eyeball its cross-section on a date, and check coverage and signs.

**Prerequisite:** `make data`. Size needs `make shares`; value/quality need
`make fundamentals` (both optional, and skipped gracefully if absent).

In [1]:
from qer.data.loader import DataLoader
from qer.factors import all_factors, get_factor, compute_factor_panel
import pandas as pd

loader = DataLoader()
[f.name for f in all_factors()], {f.name: f.direction for f in all_factors()}

(['amihud_illiquidity',
  'momentum_12_1',
  'quality_gp',
  'reversal_1m',
  'size',
  'idio_skew_60d',
  'value_btm',
  'volatility_60d'],
 {'amihud_illiquidity': 1,
  'momentum_12_1': 1,
  'quality_gp': 1,
  'reversal_1m': -1,
  'size': -1,
  'idio_skew_60d': -1,
  'value_btm': 1,
  'volatility_60d': -1})

## One factor, one date

`compute(loader, as_of)` returns the raw cross-section on a date (it slices the
panel). Values are the raw economic quantity; orientation by `direction` is the
harness's job. NaN means "no signal" (insufficient history / not yet listed) -
never a misleading zero.

In [2]:
as_of = loader.close.index[-1]
mom = get_factor("momentum_12_1").compute(loader, as_of)
mom.describe()

count    638.000000
mean       0.049641
std        0.359949
min       -3.344685
25%       -0.112189
50%        0.050777
75%        0.225578
max        2.068013
Name: momentum_12_1, dtype: float64

## The efficient path: whole panels at once

The evaluation harness never loops per date. `compute_factor_panel` calls the
factor's vectorised `compute_panel` **once** - rolling/shift transforms compute
every date in a single pass, and are look-ahead-safe by construction (row t uses
only rows <= t).

In [3]:
panel = compute_factor_panel(loader, get_factor("momentum_12_1"))
panel.shape, panel.index.min(), panel.index.max()

((4528, 848),
 Timestamp('2008-01-02 00:00:00'),
 Timestamp('2025-12-30 00:00:00'))

## Coverage and distribution across the price-only factors

For each price/volume factor, how many names have a value on the latest date,
and what does the cross-section look like. These five work on current data.

In [4]:
import numpy as np
price_only = ["momentum_12_1","reversal_1m","volatility_60d","amihud_illiquidity","idio_skew_60d"]
summary = {}
for name in price_only:
    s = get_factor(name).compute(loader, as_of)
    summary[name] = {"coverage": int(s.notna().sum()), "mean": s.mean(), "std": s.std()}
pd.DataFrame(summary).T

/home/alex/Desktop/quant-equity-research/.venv/lib/python3.12/site-packages/pandas/core/window/rolling.py:611: RuntimeWarning: All-NaN slice encountered
  return func(x, start, end, min_periods, *numba_args)


,coverage,mean,std
momentum_12_1,638.0,0.049641,0.359949
reversal_1m,646.0,0.009503,0.153041
volatility_60d,646.0,0.021997,0.026290
amihud_illiquidity,0.0,NaN,NaN
idio_skew_60d,642.0,-0.020190,1.499411


## Data-gated factors

`size` needs shares outstanding (`make shares`); `value_btm` and `quality_gp`
need ingested fundamentals (`make fundamentals`). If the data is absent these
raise a clear `FileNotFoundError` - which is exactly how `run_factor_zoo` knows
to skip them rather than fail.

In [5]:
for name in ["size","value_btm","quality_gp"]:
    try:
        s = get_factor(name).compute(loader, as_of)
        print(f"{name}: coverage {s.notna().sum()}")
    except FileNotFoundError as e:
        print(f"{name}: not available -> {str(e)[:70]}")

size: coverage 596
value_btm: coverage 0
quality_gp: coverage 429


Next: `04_factor_evaluation` runs all of these through the shared IC /
portfolio / multiple-testing harness to see which actually predict returns.